# 3.3 混合专家架构（Mixture of Experts, MoE）

## 什么是MoE？

将FFN层替换为多个专家网络（experts），通过路由器（router）为每个token选择Top-K个专家。总参数量大但每次推理仅激活部分专家，实现"大模型能力、小模型成本"。

## 为什么用MoE？

1. **解耦参数量与计算量**：MoE模型可以有数十亿甚至万亿参数，但每次推理只激活其中一小部分（如2/8专家），实际计算量远小于同等参数量的稠密模型。
2. **训练效率**：相同计算预算下，MoE模型比稠密模型能达到更好的性能，因为参数更多但计算量相同。
3. **专业化**：不同专家可以学习处理不同类型的输入，实现隐式的任务分解。

## MoE的数学原理

标准FFN：$y = \text{FFN}(x)$

MoE层：
$$y = \sum_{i \in \text{TopK}} g_i(x) \cdot E_i(x)$$

路由器计算：
$$g(x) = \text{softmax}(W_g x), \quad W_g \in \mathbb{R}^{N_e \times d}$$

其中：
- $E_i$：第$i$个专家网络（通常是FFN）
- $g_i(x)$：路由器为第$i$个专家分配的权重
- $W_g$：路由器权重矩阵
- $N_e$：专家总数
- Top-K：只选择路由权重最大的K个专家（通常K=1或2）
- 未被选中的专家不参与计算，实现稀疏激活

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import gc

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")

---
### 稀疏MoE（Sparse MoE）

每个token仅激活Top-1或Top-2专家，如Mixtral 8x7B每次仅激活2个专家（约13B参数），但拥有46B总参数的知识容量。

In [ ]:
class Expert(nn.Module):
    """单个FFN专家"""
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class MoERouter(nn.Module):
    """MoE路由器：为每个token选择Top-K专家"""
    __constants__ = ['top_k', 'n_experts']
    def __init__(self, dim, n_experts, top_k=2, noise_std=0.1):
        super().__init__()
        self.top_k = top_k
        self.n_experts = n_experts
        self.noise_std = noise_std
        self.gate = nn.Linear(dim, n_experts, bias=False)

    def forward(self, x):
        logits = self.gate(x)
        if self.training and self.noise_std > 0:
            noise = torch.randn_like(logits) * self.noise_std
            logits = logits + noise
        top_k_logits, top_k_indices = logits.topk(self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_logits, dim=-1)
        return top_k_weights, top_k_indices

class MoELayer(nn.Module):
    """稀疏MoE层"""
    def __init__(self, dim, hidden_dim, n_experts=8, top_k=2):
        super().__init__()
        self.dim = dim
        self.n_experts = n_experts
        self.top_k = top_k
        self.experts = nn.ModuleList([Expert(dim, hidden_dim) for _ in range(n_experts)])
        self.router = MoERouter(dim, n_experts, top_k)

    def forward(self, x):
        B, N, D = x.shape
        x_flat = x.reshape(-1, D)
        weights, indices = self.router(x_flat)

        output = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            expert_idx = indices[:, k]
            expert_weight = weights[:, k].unsqueeze(-1)
            for e in range(self.n_experts):
                mask = (expert_idx == e)
                if mask.any():
                    expert_input = x_flat[mask]
                    expert_output = self.experts[e](expert_input)
                    output[mask] += expert_output * expert_weight[mask]

        return output.reshape(B, N, D)

    def compute_load_balance_loss(self, router_logits):
        """负载均衡损失：鼓励各专家被均匀选择
        标准实现：f_i = 被选中频率, P_i = 平均路由概率
        注意：必须传入原始logits（softmax之前），不能传入softmax后的权重"""
        P_i = torch.softmax(router_logits, dim=-1).mean(dim=0)
        top_k_logits, top_k_indices = router_logits.topk(self.top_k, dim=-1)
        n_tokens = top_k_indices.shape[0]
        expert_counts = torch.zeros(self.n_experts, device=router_logits.device)
        for k in range(self.top_k):
            for e in range(self.n_experts):
                expert_counts[e] += (top_k_indices[:, k] == e).float().sum()
        f_i = expert_counts / (n_tokens * self.top_k)
        balance_loss = self.n_experts * (f_i * P_i).sum()
        return balance_loss

dim, hidden_dim = 128, 512
n_experts, top_k = 8, 2

moe = MoELayer(dim, hidden_dim, n_experts=n_experts, top_k=top_k)
dense_ffn = nn.Sequential(nn.Linear(dim, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, dim))

x = torch.randn(2, 32, dim)
with torch.no_grad():
    out_moe = moe(x)
    out_dense = dense_ffn(x)

moe_total_params = sum(p.numel() for p in moe.parameters())
expert_ffn_params = dim * hidden_dim + hidden_dim * dim + dim * hidden_dim
moe_active_params = expert_ffn_params * top_k + dim * n_experts
dense_params = sum(p.numel() for p in dense_ffn.parameters())

print(f"=== 稀疏MoE vs 稠密FFN ===")
print(f"MoE总参数: {moe_total_params:,}")
print(f"MoE激活参数(top-{top_k}): ~{moe_active_params:,}")
print(f"稠密FFN参数: {dense_params:,}")
print(f"MoE总参数/稠密: {moe_total_params/dense_params:.1f}x")
print(f"MoE激活参数/稠密: {moe_active_params/dense_params:.2f}x")
print(f"\n输出形状: MoE={out_moe.shape}, Dense={out_dense.shape}")

### MoE路由分析与负载均衡

#### 什么是MoE路由？

路由器为每个token选择Top-K个专家，路由权重由$g(x) = 	ext{softmax}(W_g x)$决定。负载均衡确保各专家被均匀使用。

#### 为什么需要负载均衡？

1. **路由崩塌**：不加约束时，路由器倾向于将所有token分配给少数专家，其余专家闲置
2. **训练效率**：闲置专家的参数不被训练，浪费模型容量
3. **推理效率**：负载不均导致某些专家成为瓶颈

#### 负载均衡的数学原理

**辅助损失（Load Balancing Loss）**：
$$L_{	ext{aux}} = \alpha \cdot \sum_{i=1}^{N} f_i \cdot p_i$$

其中：
- $f_i = \frac{	ext{count}(	ext{expert}_i)}{T}$：专家$i$被选中的频率
- $p_i = \frac{1}{T} \sum_{t=1}^{T} 	ext{softmax}(W_g x_t)_i$：专家$i$的平均路由概率
- $\alpha$：辅助损失权重（通常0.01）
- 当所有专家均匀使用时$f_i = p_i = 1/N$，$L_{	ext{aux}}$最小

In [ ]:
moe.eval()
x = torch.randn(4, 64, dim)
with torch.no_grad():
    x_flat = x.reshape(-1, dim)
    weights, indices = moe.router(x_flat)

expert_counts = torch.zeros(n_experts)
for k in range(top_k):
    for e in range(n_experts):
        expert_counts[e] += (indices[:, k] == e).sum().item()

total_routing = expert_counts.sum()
ideal_per_expert = total_routing / n_experts

print(f"=== MoE路由分析 ===")
print(f"总token数: {x_flat.shape[0]}")
print(f"每token选择top-{top_k}专家")
print(f"\n各专家负载:")
for e in range(n_experts):
    load = expert_counts[e].item()
    ratio = load / ideal_per_expert * 100
    bar = '█' * int(ratio / 5)
    print(f"  Expert {e}: {load:.0f} tokens ({ratio:.1f}%) {bar}")

load_std = expert_counts.std().item()
load_cv = load_std / ideal_per_expert * 100
print(f"\n负载变异系数: {load_cv:.1f}%")
print(f"理想值: 0% (完全均衡)")

logits = moe.router.gate(x_flat)
balance_loss = moe.compute_load_balance_loss(logits)
print(f"负载均衡损失: {balance_loss:.4f}")
print(f"\n负载均衡的重要性: 防止路由崩塌(routing collapse)")
print(f"  - 路由崩塌: 所有token都被路由到少数专家，其余专家闲置")
print(f"  - 解决方案: 负载均衡损失 + 噪声注入 + 容量因子")

### 容量因子与高级路由策略

#### 容量因子（Capacity Factor）

容量因子控制每个专家能处理的最大token数，防止某些专家过载：
$$\text{capacity}_i = \lfloor \frac{T \cdot K}{N_e} \cdot C_f \rfloor$$

其中$C_f$为容量因子（通常1.0-1.5），$T$为token总数，$K$为top-k，$N_e$为专家数。

- $C_f < 1.0$：某些token可能被丢弃（dropped），影响精度
- $C_f = 1.0$：恰好均匀分配，无丢弃
- $C_f > 1.0$：允许一定不均衡，但增加计算量

#### Expert Choice Routing（Google 2022）

传统Token Choice：token选专家（可能崩塌）
Expert Choice：专家选token（保证均衡）

$$\text{Expert Choice}: \text{每个专家选择top-}\frac{T \cdot K}{N_e}\text{个token}$$

优势：
1. 天然负载均衡——每个专家处理相同数量的token
2. 无需辅助损失——不需要load balance loss
3. 更高训练效率——所有专家参数都被充分利用

劣势：
1. 某些token可能被多个专家处理，某些token可能不被处理
2. 不适合自回归生成——因果模型中未来token不可见

In [ ]:
class CapacityFactorRouter(nn.Module):
    """带容量因子的路由器"""
    def __init__(self, dim, n_experts, top_k=2, capacity_factor=1.25):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.capacity_factor = capacity_factor
        self.gate = nn.Linear(dim, n_experts, bias=False)

    def forward(self, x):
        B, N, D = x.shape
        n_tokens = B * N
        x_flat = x.reshape(-1, D)
        logits = self.gate(x_flat)
        scores = torch.softmax(logits, dim=-1)
        topk_scores, topk_indices = scores.topk(self.top_k, dim=-1)

        expert_capacity = int(n_tokens * self.top_k / self.n_experts * self.capacity_factor)

        expert_counts = torch.zeros(self.n_experts, dtype=torch.long, device=x.device)
        dispatched_mask = torch.ones(n_tokens, self.top_k, dtype=torch.bool, device=x.device)
        dropped_count = 0

        for t in range(n_tokens):
            for k in range(self.top_k):
                e = topk_indices[t, k].item()
                if expert_counts[e] >= expert_capacity:
                    dispatched_mask[t, k] = False
                    dropped_count += 1
                else:
                    expert_counts[e] += 1

        return topk_scores, topk_indices, dispatched_mask, dropped_count, expert_counts

class ExpertChoiceRouter(nn.Module):
    """Expert Choice路由：专家选token，天然均衡"""
    def __init__(self, dim, n_experts, top_k=2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.gate = nn.Linear(dim, n_experts, bias=False)

    def forward(self, x):
        B, N, D = x.shape
        n_tokens = B * N
        x_flat = x.reshape(-1, D)
        logits = self.gate(x_flat)
        scores = torch.softmax(logits, dim=-1)

        tokens_per_expert = n_tokens * self.top_k // self.n_experts

        expert_assignments = []
        for e in range(self.n_experts):
            expert_scores = scores[:, e]
            top_scores, top_indices = expert_scores.topk(min(tokens_per_expert, n_tokens))
            expert_assignments.append((top_indices, top_scores))

        return expert_assignments, tokens_per_expert

dim, n_experts, top_k = 128, 8, 2
cf_router = CapacityFactorRouter(dim, n_experts, top_k, capacity_factor=1.25)
ec_router = ExpertChoiceRouter(dim, n_experts, top_k)

x = torch.randn(4, 64, dim)
with torch.no_grad():
    scores, indices, mask, dropped, counts = cf_router(x)
    ec_assignments, tpe = ec_router(x)

print(f"=== 容量因子路由 (C_f=1.25) ===")
print(f"总token数: {4*64}")
print(f"专家容量: {int(4*64*2/8*1.25)}")
print(f"丢弃token数: {dropped}")
print(f"各专家负载: {counts.tolist()}")

print(f"\n=== Expert Choice路由 ===")
print(f"每专家处理token数: {tpe}")
for e, (idx, sc) in enumerate(ec_assignments):
    print(f"  Expert {e}: 处理{len(idx)}个token, 平均分数={sc.mean():.4f}")

print(f"\n路由策略对比:")
print(f"  {'策略':<25} {'负载均衡':<12} {'丢弃率':<10} {'适用场景':<20}")
print(f"  {'-'*67}")
print(f"  {'Token Choice+Aux Loss':<25} {'需辅助损失':<12} {'低':<10} {'通用':<20}")
print(f"  {'Capacity Factor':<25} {'容量限制':<12} {'中':<10} {'训练/批推理':<20}")
print(f"  {'Expert Choice':<25} {'天然均衡':<12} {'0%':<10} {'非因果模型训练':<20}")

### MoE量化与端侧部署优化

#### 什么是MoE量化？

对MoE模型中的专家权重进行量化，减少模型大小和推理内存。由于MoE总参数量大，量化收益更显著。

#### 为什么MoE量化更难？

1. **专家权重分布不同**：不同专家学到的模式不同，权重分布差异大
2. **路由动态性**：不同输入激活不同专家，难以统一量化策略
3. **内存碎片**：MoE的稀疏激活导致内存访问不连续

#### MoE量化的数学原理

逐专家量化：对第$i$个专家的权重$W_i$独立计算量化参数：
$$s_i = \frac{\max(|W_i|)}{q_{\max}}, \quad W_{q,i} = 	ext{round}(W_i / s_i)$$

端侧部署优化：
1. **专家卸载**：将不常用的专家存储在慢速内存，按需加载
2. **专家共享**：相似专家共享量化参数，减少存储
3. **动态精度**：根据路由频率分配精度（高频专家INT8，低频专家INT4）

In [ ]:
def quantize_moe_experts(moe_layer, n_bits=4):
    """量化MoE专家权重"""
    q_max = 2 ** (n_bits - 1) - 1
    q_min = -2 ** (n_bits - 1)
    quantized_experts = {}

    for e_idx, expert in enumerate(moe_layer.experts):
        expert_q = {}
        for name, param in expert.named_parameters():
            scale = param.abs().max() / q_max
            scale = torch.clamp(scale, min=1e-8)
            q_weights = torch.clamp(torch.round(param / scale), q_min, q_max)
            expert_q[name] = {'q': q_weights, 'scale': scale}
        quantized_experts[e_idx] = expert_q

    return quantized_experts

def compute_moe_expert_similarity(moe_layer):
    """计算MoE专家间的权重相似度"""
    n = len(moe_layer.experts)
    similarities = torch.zeros(n, n)
    for i in range(n):
        for j in range(n):
            w_i = torch.cat([p.flatten() for p in moe_layer.experts[i].parameters()])
            w_j = torch.cat([p.flatten() for p in moe_layer.experts[j].parameters()])
            similarities[i, j] = F.cosine_similarity(w_i.unsqueeze(0), w_j.unsqueeze(0)).item()
    return similarities

sim = compute_moe_expert_similarity(moe)
print(f"=== MoE专家间权重相似度 ===")
print(f"平均相似度: {sim.mean():.4f}")
print(f"最大相似度: {sim[sim < 1.0].max():.4f}")
print(f"最小相似度: {sim[sim > 0].min():.4f}")

q_experts = quantize_moe_experts(moe, n_bits=4)

fp16_mem = sum(p.numel() * 2 for p in moe.parameters()) / 1024
int4_mem = sum(
    v['q'].numel() * 0.5 + v['scale'].numel() * 2
    for expert_q in q_experts.values()
    for v in expert_q.values()
) / 1024

print(f"\n=== MoE量化存储对比 ===")
print(f"FP16专家权重: {fp16_mem:.2f} KB")
print(f"INT4量化权重: {int4_mem:.2f} KB")
print(f"压缩比: {fp16_mem/int4_mem:.1f}x")

print(f"\n=== MoE端侧部署策略 ===")
print(f"1. 按需加载专家: 仅加载被激活的专家权重到内存")
print(f"2. 专家量化: 不活跃专家可更激进量化(INT4/INT2)")
print(f"3. 专家共享: 相似度高的专家可合并")
print(f"4. 专家剪枝: 移除极少被选中的专家")

print(f"\n=== 产业级MoE量化方法 ===")
print(f"1. QMoE: 逐专家GPTQ量化，每专家独立校准，精度损失最小")
print(f"2. AWQ-MoE: 基于激活感知的逐专家量化，保护重要权重通道")
print(f"3. 动态精度路由: 高频专家INT8(精度优先)，低频专家INT4(存储优先)")
print(f"4. 专家卸载量化: 卸载到磁盘的专家用INT2/INT3极致压缩")
print(f"5. DeepSeek-MoE实践: 共享专家+路由专家分离，共享专家保持高精度")

### 产业级实战：高效 MoE 前向传播（scatter/gather）

产业界 MoE 实现使用 scatter/gather 操作替代 for 循环，实现批量并行计算。以下对比朴素循环 vs 高效实现。

#### scatter/gather 的核心思路

1. **Scatter**：根据路由结果，将token分配到对应的专家
2. **并行计算**：每个专家对分配到的所有token执行一次批量矩阵乘
3. **Gather**：将各专家的输出按原始顺序合并

#### 为什么scatter/gather更高效？

- 朴素循环：逐token调用专家，GPU利用率极低（<5%）
- scatter/gather：批量并行计算，GPU利用率高（>80%）

#### 数学原理

设$S_i$为分配给专家$i$的token索引集合：
$$Y_{S_i} = E_i(X_{S_i}), \quad i = 1, \ldots, N_e$$
一次矩阵乘完成所有分配给专家$i$的token计算。

In [ ]:
import time

class EfficientMoELayer(nn.Module):
    """高效MoE: 使用scatter/gather替代for循环"""
    def __init__(self, dim, n_experts=8, top_k=2, ffn_dim=None):
        super().__init__()
        self.dim = dim
        self.n_experts = n_experts
        self.top_k = top_k
        ffn_dim = ffn_dim or dim * 4

        self.gate = nn.Linear(dim, n_experts, bias=False)
        self.experts_up = nn.Parameter(torch.randn(n_experts, ffn_dim, dim) * 0.02)
        self.experts_down = nn.Parameter(torch.randn(n_experts, dim, ffn_dim) * 0.02)

    def forward(self, x):
        B, N, D = x.shape
        x_flat = x.reshape(-1, D)
        n_tokens = x_flat.shape[0]

        logits = self.gate(x_flat)
        scores = torch.softmax(logits, dim=-1)
        topk_scores, topk_indices = scores.topk(self.top_k, dim=-1)

        output = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            expert_indices = topk_indices[:, k]
            expert_scores = topk_scores[:, k].unsqueeze(-1)

            up_weights = self.experts_up[expert_indices]
            down_weights = self.experts_down[expert_indices]

            hidden = torch.bmm(x_flat.unsqueeze(1), up_weights.transpose(1, 2)).squeeze(1)
            hidden = F.silu(hidden)
            expert_out = torch.bmm(hidden.unsqueeze(1), down_weights.transpose(1, 2)).squeeze(1)

            output = output + expert_scores * expert_out

        return output.reshape(B, N, D)

class NaiveMoELayer(nn.Module):
    """朴素MoE: for循环逐专家计算"""
    def __init__(self, dim, n_experts=8, top_k=2, ffn_dim=None):
        super().__init__()
        self.dim = dim
        self.n_experts = n_experts
        self.top_k = top_k
        ffn_dim = ffn_dim or dim * 4

        self.gate = nn.Linear(dim, n_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, ffn_dim), nn.GELU(), nn.Linear(ffn_dim, dim))
            for _ in range(n_experts)
        ])

    def forward(self, x):
        B, N, D = x.shape
        x_flat = x.reshape(-1, D)

        logits = self.gate(x_flat)
        scores = torch.softmax(logits, dim=-1)
        topk_scores, topk_indices = scores.topk(self.top_k, dim=-1)

        output = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            for e in range(self.n_experts):
                mask = (topk_indices[:, k] == e)
                if mask.any():
                    expert_input = x_flat[mask]
                    expert_out = self.experts[e](expert_input)
                    output[mask] += topk_scores[mask, k:k+1] * expert_out

        return output.reshape(B, N, D)

dim = 128
naive_moe = NaiveMoELayer(dim, n_experts=8, top_k=2)
efficient_moe = EfficientMoELayer(dim, n_experts=8, top_k=2)
naive_moe.eval()
efficient_moe.eval()

x = torch.randn(4, 32, dim)

with torch.no_grad():
    for _ in range(5):
        naive_moe(x)
    start = time.perf_counter()
    for _ in range(20):
        naive_moe(x)
    naive_time = (time.perf_counter() - start) / 20 * 1000

    for _ in range(5):
        efficient_moe(x)
    start = time.perf_counter()
    for _ in range(20):
        efficient_moe(x)
    efficient_time = (time.perf_counter() - start) / 20 * 1000

naive_params = sum(p.numel() for p in naive_moe.parameters())
efficient_params = sum(p.numel() for p in efficient_moe.parameters())

print(f"=== 高效MoE vs 朴素MoE ===")
print(f"配置: {8}专家, top-{2}, dim={dim}")
print(f"\n{'实现':<20} {'参数量':<15} {'延迟(ms)':<12} {'加速比':<10}")
print("-" * 57)
print(f"{'朴素(for循环)':<20} {naive_params:<15,} {naive_time:<12.2f} {'1.00x':<10}")
print(f"{'高效(bmm/scatter)':<20} {efficient_params:<15,} {efficient_time:<12.2f} {naive_time/efficient_time:.2f}x")

print(f"\n产业界MoE实现方式:")
print(f"1. Megablocks: 基于稀疏矩阵的grouped GEMM (GPU最优)")
print(f"2. Scatter/Gather: bmm批量计算 (本实现, 通用性好)")
print(f"3. Expert Parallelism: All-to-All通信跨GPU分发 (多GPU训练)")
print(f"4. 端侧优化: 按需加载专家权重 + 路由预测预取")
print(f"5. 框架: Mixtral (HuggingFace), DeepSpeed-MoE, Megablocks")

del naive_moe, efficient_moe
gc.collect()